# Hands-on: Hajj Crowd Digital Twin
### IoT in Large Scale Events | Google Colab Free

**TIC06 Internet of Things | Praktikum pendamping PPT 50 slide (terutama slide 41–47)**

Notebook ini adalah **buku praktikum sekaligus simulator IoT**, bukan sistem pemantauan Haji yang digunakan untuk operasi sebenarnya. Seluruh perangkat, jemaah, data lokasi, kapasitas, dan kejadian bersifat **sintetis**. Lokasi pada peta hanya pendekatan geografis untuk latihan; jangan menggunakannya sebagai rute, kapasitas, atau acuan keselamatan operasional.

**Alur praktikum:** `virtual wristband → publisher → MQTT broker → subscriber → Digital Twin state → geospatial analytics → dashboard & alert`.

**CPMK:** Menerapkan konsep IoT untuk merancang sistem pemantauan dan pengelolaan kerumunan pada acara berskala besar.

**Sub-CPMK:** (1) merancang telemetry perangkat dan komunikasi publish/subscribe; (2) membuat representasi lokasi dan state Digital Twin; (3) menghitung occupancy serta geofence; (4) mengembangkan dashboard dan alert; (5) mengevaluasi data freshness, throughput, dan keterbatasan pengukuran.

**Cara pakai:** jalankan sel **berurutan dari atas** (Runtime → Run all). Gunakan satu sesi simulator pada satu waktu. Sel konfigurasi boleh diubah kemudian jalankan ulang bagian setup dan reset simulator. Runtime Colab bersifat sementara; hentikan simulasi sebelum memutus runtime.

## 1. Konsep: mengapa bukan sekadar IoT dashboard?

- **IoT sensing:** setiap wristband menghasilkan observasi (`device_id`, koordinat, status, timestamp), bukan langsung keputusan.
- **MQTT:** memindahkan observasi dari publisher ke subscriber melalui broker menggunakan **topic**. MQTT **bukan** teknologi penentuan lokasi.
- **Digital Twin:** merepresentasikan entitas (wristband, zona, koridor), hubungan, dan *state* terkini. State diperbarui hanya dari pesan yang berhasil diterima dan divalidasi.
- **Spatial context:** posisi diinterpretasikan terhadap batas zona, luas area, jalur, dan kapasitas simulasi.
- **Decision Support System:** aturan memberi sinyal untuk **pemeriksaan operator**, bukan keputusan otomatis tentang keselamatan.

**Catatan metodologis:** sistem ini merupakan *state-based operational Digital Twin prototype*. Belum menjalankan model fisika kerumunan, simulasi evakuasi tervalidasi, atau high-fidelity 3D. Sejalan dengan Greg Biegel, *Digital Twins in Action*, bagian representasi, sensing, data integration, spatial context, visualization, dan simulation.

## 2. Setup library dan pemeriksaan lingkungan

Sel berikut akan memasang pustaka hanya jika belum tersedia. **Colab Free CPU cukup** untuk skenario dasar. Internet dibutuhkan untuk instalasi paket, peta basemap, dan broker eksternal. Broker lokal MQTT dapat dijalankan dalam runtime apabila paket `amqtt` berhasil terpasang.

**Mode komunikasi yang disediakan:**
1. `local_mqtt`: broker MQTT di dalam runtime Colab menggunakan `amqtt`, dengan klien `paho-mqtt`; pilihan utama agar tidak bergantung broker publik.
2. `public_mqtt`: broker publik `test.mosquitto.org:1883`, hanya untuk data sintetik dan topic acak; mungkin diblokir jaringan dan tidak menjamin privasi/ketersediaan.
3. `local_queue`: simulasi publish/subscribe in-process **bukan protokol MQTT**; fallback pembelajaran ketika jaringan/paket MQTT gagal. Dashboard dan Digital Twin tetap berfungsi, tetapi mahasiswa harus menyatakan bahwa pengujian MQTT tidak berhasil.

In [ ]:
import importlib.util, subprocess, sys
packages = {"paho.mqtt": "paho-mqtt>=2.1,<3", "amqtt": "amqtt>=0.11,<1", "ipywidgets":"ipywidgets>=8,<9", "folium":"folium>=0.16", "shapely":"shapely>=2", "pandas":"pandas>=2", "matplotlib":"matplotlib>=3"}
missing = [p for module,p in packages.items() if importlib.util.find_spec(module.split('.')[0]) is None]
if missing:
    print("Installing:", missing)
    result = subprocess.run([sys.executable,"-m","pip","install","-q",*missing], capture_output=True, text=True)
    if result.returncode != 0:
        print("PERINGATAN: Instalasi tidak sepenuhnya berhasil. Cuplikan error:", result.stderr[-1300:])
else:
    print("Semua paket utama tersedia.")
print("Python:", sys.version.split()[0])

Semua paket utama tersedia.
Python: 3.13.18


## 3. Configuration: parameter

Terdapat **empat zona simulasi** yang tersusun dari barat ke timur. Luas dan kapasitas hanyalah angka latihan. Pengunjung virtual bergerak dalam koridor terdefinisi, bukan melompat ke koordinat acak tanpa rute.

**Eksperimen awal:** 100 wristband; satu telemetry per perangkat setiap 2 detik (rata-rata teoretis 50 pesan/detik). Pesan dikirim merata selama interval untuk mengurangi burst. `TIME_SCALE` mempercepat pergerakan *waktu simulasi*, bukan mempercepat jam nyata.

In [ ]:
from dataclasses import dataclass
from datetime import datetime, timezone
from collections import deque, defaultdict
import math, random, json, uuid, time, threading, queue, socket, asyncio, statistics
import numpy as np, pandas as pd
from IPython.display import display, clear_output, Markdown

@dataclass
class Config:
    N_DEVICES: int = 100
    TELEMETRY_INTERVAL_S: float = 2.0
    TIME_SCALE: float = 20.0
    GPS_NOISE_M: float = 3.0
    SCENARIO: str = 'normal'  # normal | crowded_B | device_offline
    TRANSPORT: str = 'local_mqtt'  # local_mqtt | public_mqtt | local_queue
    STALE_AFTER_S: float = 8.0
    RUN_SECONDS: int = 40
    MAP_REFRESH_S: float = 4.0
    RANDOM_SEED: int = 42
    MQTT_PORT: int = 18884  # broker lokal, hindari 1883 yang mungkin terpakai

CFG = Config()
assert CFG.N_DEVICES > 0 and CFG.TELEMETRY_INTERVAL_S > 0
assert CFG.SCENARIO in ('normal','crowded_B','device_offline')
SESSION_ID = uuid.uuid4().hex[:10]
MQTT_TOPIC = f'ubm/tic06/hajj-sim/{SESSION_ID}/telemetry'
print('Session:', SESSION_ID, '| Topic:', MQTT_TOPIC)
print('Expected average messages/s:', round(CFG.N_DEVICES / CFG.TELEMETRY_INTERVAL_S, 1))

Session: 260b9f47a6 | Topic: ubm/tic06/hajj-sim/260b9f47a6/telemetry
Expected average messages/s: 50.0


## 4. Spatial representation: zona, jalur, dan georeferensi

Sebuah **Digital Twin memerlukan model lingkungannya**, bukan hanya daftar posisi perangkat. Kita memakai koordinat lokal `(x,y)` dalam meter untuk menghitung lintasan dan area. Transformasi ke latitude-longitude hanya untuk visualisasi geospatial. Skala konversi menggunakan pendekatan lokal yang cukup untuk contoh kecil, **bukan surveying**.

**Zona:** A (masuk), B (koridor), C (transit), D (keluar). Masing-masing berbentuk polygon sederhana sepanjang 250 meter dan lebar 160 meter. Pada proyek nyata, geometry perlu diperoleh dari GIS yang diverifikasi.

In [ ]:
# Pusat koordinat ilustratif di dekat Mina, bukan penanda lokasi fasilitas yang sebenarnya.
ORIGIN_LAT, ORIGIN_LON = 21.414, 39.894
M_PER_DEG_LAT = 111_320.0
M_PER_DEG_LON = 111_320.0 * math.cos(math.radians(ORIGIN_LAT))

def xy_to_latlon(x, y):
    return ORIGIN_LAT + y/M_PER_DEG_LAT, ORIGIN_LON + x/M_PER_DEG_LON

def latlon_to_xy(lat, lon):
    return (lon-ORIGIN_LON)*M_PER_DEG_LON, (lat-ORIGIN_LAT)*M_PER_DEG_LAT

ZONES = {
 'A': {'x0':0,   'x1':250,  'y0':-80, 'y1':80, 'capacity':45, 'label':'Entry'},
 'B': {'x0':250, 'x1':500,  'y0':-80, 'y1':80, 'capacity':30, 'label':'Corridor'},
 'C': {'x0':500, 'x1':750,  'y0':-80, 'y1':80, 'capacity':45, 'label':'Transit'},
 'D': {'x0':750, 'x1':1000, 'y0':-80, 'y1':80, 'capacity':45, 'label':'Exit'},
}

def point_zone(x, y):
    # x==1000 dimasukkan ke zona D agar batas terluar tetap terhitung
    for name,z in ZONES.items():
        if z['x0'] <= x < z['x1'] and z['y0'] <= y <= z['y1']:
            return name
    if abs(x-1000)<1e-8 and -80<=y<=80:
        return 'D'
    return 'OUTSIDE'

def polygon_latlon(z):
    corners = [(z['x0'],z['y0']),(z['x1'],z['y0']),(z['x1'],z['y1']),(z['x0'],z['y1'])]
    return [xy_to_latlon(x,y) for x,y in corners]

print(pd.DataFrame([{'zone':name,'name':z['label'], 'area_m2':(z['x1']-z['x0'])*(z['y1']-z['y0']), 'training_capacity':z['capacity']} for name,z in ZONES.items()]).to_string(index=False))

zone     name  area_m2  training_capacity
   A    Entry    40000                 45
   B Corridor    40000                 30
   C  Transit    40000                 45
   D     Exit    40000                 45


## 5. Virtual wristbands: simulasi perangkat fisik

Setiap perangkat memiliki ID, posisi, kecepatan, baterai, nomor urut pesan, dan rute dalam empat zona. **Tidak ada perangkat fisik**: modul positioning dan baterai dimodelkan dengan Python. Dengan `SCENARIO='crowded_B'`, sebagian besar perangkat mulai di zona B; dengan `'device_offline'`, sebagian perangkat berhenti mengirim sesudah beberapa detik.

Perhatikan perbedaan `truth state` yang hanya dimiliki simulator dengan `observed state` yang hanya boleh dimiliki Digital Twin setelah pesan diterima.

In [ ]:
class VirtualWristband:
    def __init__(self, number, rng, scenario='normal'):
        self.device_id = f'WB-{number:03d}'
        self.rng = rng
        if scenario == 'crowded_B' and rng.random() < .70:
            self.x = rng.uniform(275, 485)
        else:
            self.x = rng.uniform(0, 995)
        self.y = rng.uniform(-53, 53)
        self.speed_mps = rng.uniform(0.7, 1.5)
        self.battery_pct = rng.uniform(60, 100)
        self.seq = 0
        self.offline = scenario == 'device_offline' and number % 10 == 0

    def step(self, dt_sim_s, noise_m=0):
        self.x = (self.x + self.speed_mps * dt_sim_s) % 1000
        self.battery_pct = max(0, self.battery_pct - 0.0001 * dt_sim_s)
        self.seq += 1
        obs_x = self.x + self.rng.gauss(0,noise_m)
        obs_y = self.y + self.rng.gauss(0,noise_m)
        lat, lon = xy_to_latlon(obs_x,obs_y)
        return {'device_id':self.device_id, 'timestamp':datetime.now(timezone.utc).isoformat(),
                'sequence_no':self.seq,'latitude':round(lat,8),'longitude':round(lon,8),
                'battery_pct':round(self.battery_pct,2),'status':'ACTIVE',
                'accuracy_m':float(noise_m), 'schema_version':1}

def make_fleet(cfg):
    rng = random.Random(cfg.RANDOM_SEED)
    return [VirtualWristband(i+1,random.Random(rng.randrange(10**9)),cfg.SCENARIO) for i in range(cfg.N_DEVICES)]

fleet_preview = make_fleet(CFG)
example_payload = fleet_preview[0].step(0,CFG.GPS_NOISE_M)
print(json.dumps(example_payload,indent=2,ensure_ascii=False))
assert point_zone(*latlon_to_xy(example_payload['latitude'], example_payload['longitude'])) in (*ZONES.keys(), 'OUTSIDE')

{
  "device_id": "WB-001",
  "timestamp": "2026-11-10T02:04:42.958153+00:00",
  "sequence_no": 1,
  "latitude": 21.41362396,
  "longitude": 39.90048639,
  "battery_pct": 86.22,
  "status": "ACTIVE",
  "accuracy_m": 3.0,
  "schema_version": 1
}


## 6. MQTT publisher, broker, dan subscriber

**MQTT = Message Queuing Telemetry Transport**, protokol publish/subscribe. Publisher tidak perlu mengetahui subscriber; keduanya berkomunikasi melalui broker dengan nama topic yang sama.

**QoS 0** dipilih untuk demonstrasi telemetry bergerak. Pesan dapat hilang dan tidak boleh diasumsikan exactly-once. Nomor urut `sequence_no` dipakai untuk menolak pesan duplikat dan out-of-order.

Broker lokal menggunakan paket `amqtt`. Bila paket tidak dapat dipasang atau broker tidak berhasil start, sistem beralih ke broker publik hanya jika `TRANSPORT='public_mqtt'` disetel secara eksplisit. Mode `local_queue` tetap tersedia untuk menyelesaikan bagian Digital Twin tanpa MQTT. Jangan kirim identitas atau lokasi orang sungguhan ke broker publik.

In [ ]:
# Periksa availability library MQTT
try:
    import paho.mqtt.client as mqtt
    PAHO_AVAILABLE=True
except Exception as exc:
    mqtt=None; PAHO_AVAILABLE=False
    print('Paho MQTT tidak tersedia:', type(exc).__name__)
try:
    from amqtt.broker import Broker
    AMQTT_AVAILABLE=True
except Exception as exc:
    Broker=None; AMQTT_AVAILABLE=False
    print('amqtt broker tidak tersedia:', type(exc).__name__)
print('MQTT client:',PAHO_AVAILABLE, '| MQTT broker lokal:',AMQTT_AVAILABLE)

MQTT client: True | MQTT broker lokal: True


In [ ]:
# Menjalankan server MQTT internal di dalam google colab
class LocalMQTTBroker:
    """Broker MQTT 3.1.1 untuk praktikum pada 127.0.0.1 (tidak terekspos Internet)."""
    def __init__(self, port):
        self.port=port; self.thread=None; self.loop=None; self.broker=None; self.error=None
    def start(self, timeout=8):
        if not AMQTT_AVAILABLE:
            raise RuntimeError('Paket amqtt belum tersedia')
        ready=threading.Event()
        def worker():
            self.loop=asyncio.new_event_loop(); asyncio.set_event_loop(self.loop)
            try:
                cfg={'listeners':{'default':{'type':'tcp','bind':f'127.0.0.1:{self.port}'}},
                     'sys_interval':0,'timeout_disconnect_delay':2,
                     'plugins':{'amqtt.plugins.authentication.AnonymousAuthPlugin':{'allow_anonymous':True}}}
                self.broker=Broker(config=cfg)
                self.loop.run_until_complete(self.broker.start())
                ready.set(); self.loop.run_forever()
            except Exception as exc:
                self.error=exc; ready.set()
            finally:
                if self.broker is not None:
                    try: self.loop.run_until_complete(self.broker.shutdown())
                    except Exception: pass
                self.loop.close()
        self.thread=threading.Thread(target=worker,daemon=True)
        self.thread.start()
        if not ready.wait(timeout): raise TimeoutError('Broker lokal timeout')
        if self.error: raise RuntimeError(str(self.error))
        return self
    def stop(self):
        if self.loop and self.loop.is_running():
            self.loop.call_soon_threadsafe(self.loop.stop)
            if self.thread: self.thread.join(timeout=5)

# Transport lokal tanpa MQTT: digunakan HANYA sebagai cadangan
class LocalQueueTransport:
    name='local_queue (BUKAN MQTT)'
    def __init__(self, receive): self.receive=receive; self.queue=queue.Queue(); self.running=False
    def start(self):
        self.running=True
        def worker():
            while self.running or not self.queue.empty():
                try: packet=self.queue.get(timeout=.2)
                except queue.Empty: continue
                self.receive(packet, time.time())
        self.thread=threading.Thread(target=worker,daemon=True); self.thread.start()
    def publish(self,payload):
        if self.running: self.queue.put(json.dumps(payload))
    def stop(self):
        self.running=False
        if hasattr(self,'thread'): self.thread.join(timeout=3)
# Mengelola pengiriman dan penerimaan pesan telemetri berbasis MQTT
class PahoMQTTTransport:
    name='MQTT QoS 0'
    def __init__(self,receive,host,port,topic):
        if not PAHO_AVAILABLE: raise RuntimeError('paho-mqtt diperlukan')
        self.receive=receive; self.host=host; self.port=port; self.topic=topic
        self.sub=mqtt.Client(mqtt.CallbackAPIVersion.VERSION2,client_id=f'ubm-sub-{uuid.uuid4().hex[:8]}')
        self.pub=mqtt.Client(mqtt.CallbackAPIVersion.VERSION2,client_id=f'ubm-pub-{uuid.uuid4().hex[:8]}')
        self.connected=threading.Event(); self.subscribed=threading.Event()
        def on_connect(client,userdata,flags,reason_code,properties):
            if reason_code==0:
                self.connected.set(); client.subscribe(self.topic,qos=0)
        def on_subscribe(client,userdata,mid,reason_codes,properties): self.subscribed.set()
        def on_message(client,userdata,msg): self.receive(msg.payload.decode('utf-8'),time.time())
        self.sub.on_connect=on_connect; self.sub.on_subscribe=on_subscribe; self.sub.on_message=on_message
    def start(self):
        self.sub.connect(self.host,self.port,keepalive=30); self.sub.loop_start()
        if not self.connected.wait(7) or not self.subscribed.wait(7):
            self.stop(); raise TimeoutError('MQTT subscriber gagal terhubung/subscribe')
        self.pub.connect(self.host,self.port,keepalive=30); self.pub.loop_start()
    def publish(self,payload):
        info=self.pub.publish(self.topic,json.dumps(payload),qos=0)
        if info.rc!=mqtt.MQTT_ERR_SUCCESS: raise RuntimeError(f'MQTT publish rc={info.rc}')
    def stop(self):
        for c in (self.pub,self.sub):
            try: c.disconnect(); c.loop_stop()
            except Exception: pass

## 7. Digital Twin state engine: telemetry menjadi model keadaan

Setiap pesan menjalani validasi schema, koordinat, dan sequence number. State terbaru disimpan berdasarkan `device_id`. Pesan dengan nomor urut lebih rendah/tidak berubah tidak diterapkan. Device yang tidak mengirim setelah `STALE_AFTER_S` dinyatakan **STALE**, bukan otomatis dianggap hilang atau keluar dari zona.

**Metrik:** `transport_latency_ms` mengukur waktu dari payload dibuat sampai diterima subscriber dalam satu runtime (bukan network-only latency); `state_age_s` mengukur umur observation terbaru. Keakuratan GPS dipengaruhi noise, sehingga satu titik berada di zona belum tentu mewakili posisi yang pasti.

In [ ]:
# Pemroses utama (processing engine) untuk representasi jamaah yang bergerak
class DigitalTwinEngine:
    def __init__(self,cfg):
        self.cfg=cfg; self.lock=threading.RLock(); self.states={}
        self.counts=defaultdict(int); self.latencies_ms=deque(maxlen=10000)
        self.events=deque(maxlen=12000); self.transitions=deque(maxlen=5000)
        self.created=time.time()

    def ingest(self,raw,received_at=None):
        received_at=time.time() if received_at is None else received_at
        try:
            p=json.loads(raw) if isinstance(raw,str) else dict(raw)
            required=['device_id','timestamp','sequence_no','latitude','longitude','battery_pct','status']
            if any(k not in p for k in required): raise ValueError('missing required field')
            if not isinstance(p['device_id'],str) or not p['device_id'].startswith('WB-'): raise ValueError('bad id')
            seq=int(p['sequence_no']); lat=float(p['latitude']); lon=float(p['longitude'])
            if not (-90<=lat<=90 and -180<=lon<=180): raise ValueError('invalid latlon')
            event_ts=datetime.fromisoformat(p['timestamp'].replace('Z','+00:00')).timestamp()
            if not math.isfinite(event_ts) or abs(received_at-event_ts)>86400: raise ValueError('implausible timestamp')
        except Exception:
            with self.lock: self.counts['invalid']+=1
            return False
        with self.lock:
            old=self.states.get(p['device_id'])
            if old is not None and seq<=old['sequence_no']:
                self.counts['duplicate_or_old']+=1
                return False
            x,y=latlon_to_xy(lat,lon)
            zone=point_zone(x,y)
            if old is not None and old['zone']!=zone:
                self.transitions.append({'time':received_at,'device_id':p['device_id'], 'from':old['zone'],'to':zone})
            self.states[p['device_id']]={'device_id':p['device_id'],'sequence_no':seq,
                 'latitude':lat,'longitude':lon,'x':x,'y':y,'zone':zone,
                 'battery_pct':float(p['battery_pct']),'event_ts':event_ts,
                 'received_ts':received_at,'accuracy_m':float(p.get('accuracy_m',0))}
            latency_ms=max(0.0,(received_at-event_ts)*1000)
            self.latencies_ms.append(latency_ms)
            self.events.append({'received_ts':received_at,'event_ts':event_ts,'device_id':p['device_id'],
                                'seq':seq,'zone':zone,'latency_ms':latency_ms})
            self.counts['accepted']+=1
        return True
    # hitung latensi, kepadatan zona, trigger alert
    def snapshot(self,now=None):
        now=time.time() if now is None else now
        with self.lock:
            states=[dict(v,stale=(now-v['received_ts']>self.cfg.STALE_AFTER_S),
                         state_age_s=max(0,now-v['received_ts'])) for v in self.states.values()]
            counts=dict(self.counts);lat=list(self.latencies_ms); transitions=list(self.transitions)
        active=[s for s in states if not s['stale']]
        occupancy={z:sum(s['zone']==z for s in active) for z in ZONES}
        zone_stats={z:{'devices':n,'capacity':ZONES[z]['capacity'],
                       'ratio':n/ZONES[z]['capacity'],
                       'observed_density_per_m2':n/((ZONES[z]['x1']-ZONES[z]['x0'])*(ZONES[z]['y1']-ZONES[z]['y0']))}
                    for z,n in occupancy.items()}
        alert=[]
        for z,st in zone_stats.items():
            if st['ratio']>1: alert.append(f'CHECK OCCUPANCY: Zone {z} > training capacity ({st["devices"]}/{st["capacity"]})')
        nstale=sum(s['stale'] for s in states)
        if nstale: alert.append(f'DEVICE DATA STALE: {nstale} device(s)')
        outside=sum(s['zone']=='OUTSIDE' for s in active)
        if outside: alert.append(f'CHECK GEOFENCE: {outside} device(s) outside polygons')
        return {'states':states,'zone_stats':zone_stats,'alerts':alert,'active':len(active),
                'stale':nstale,'count_received':counts.get('accepted',0),'invalid':counts.get('invalid',0),
                'old':counts.get('duplicate_or_old',0), 'latency_p50_ms':float(np.median(lat)) if lat else None,
                'latency_p95_ms':float(np.percentile(lat,95)) if lat else None,
                'transitions':transitions,'now':now}

DT = DigitalTwinEngine(CFG)
print('Digital Twin siap. State awal:',len(DT.states),'device(s)')

Digital Twin siap. State awal: 0 device(s)


## 8. Simulation controller: menghubungkan semua komponen

**Mulai dari sini untuk menjalankan sistem end-to-end.** Controller hanya mengirim payload ke `transport.publish()`. **Tidak ada kode yang memindahkan posisi simulator langsung ke DT**. Subscriber yang menerima pesan harus memanggil `DT.ingest()`.

Untuk mode MQTT, jika `local_mqtt` gagal karena versi paket, sistem **secara eksplisit memberi label fallback** ke `local_queue`. Hal itu memastikan latihan mapping dan dashboard tetap dapat dilakukan tanpa menyamakan fallback dengan MQTT.

In [ ]:
# orchestrator utama yang menyatukan seluruh komponen simulasi: menghubungkan
# Wristband, komunikasi (Transport/Broker) dan Digital Twin Engine
class SimulatorController:
    def __init__(self,cfg):
        self.cfg=cfg; self.engine=DigitalTwinEngine(cfg)
        self.devices=make_fleet(cfg);self.transport=None;self.broker=None
        self.stop_event=threading.Event();self.thread=None;self.publish_errors=0
        self.sent=0;self.started_at=None;self.mode='not started';self.last_error=None
    def _receive(self,payload,received_at):
        self.engine.ingest(payload,received_at)
    # Controller memilih opsi transport jaringan berdasar konfigurasi
    def start(self):
        if self.thread and self.thread.is_alive(): return
        self.stop_event.clear(); self.started_at=time.time()
        if self.cfg.TRANSPORT=='local_queue':
            self.transport=LocalQueueTransport(self._receive);self.transport.start();self.mode=self.transport.name
        else:
            try:
                if self.cfg.TRANSPORT=='local_mqtt':
                    self.broker=LocalMQTTBroker(self.cfg.MQTT_PORT).start()
                    host,port='127.0.0.1',self.cfg.MQTT_PORT
                elif self.cfg.TRANSPORT=='public_mqtt':
                    host,port='test.mosquitto.org',1883
                else: raise ValueError('TRANSPORT tidak dikenal')
                self.transport=PahoMQTTTransport(self._receive,host,port,MQTT_TOPIC)
                self.transport.start();self.mode=f'MQTT broker={host}:{port}'
            except Exception as exc:
                self.last_error=f'{type(exc).__name__}: {exc}'
                print('MQTT gagal:',self.last_error)
                if self.broker:
                    self.broker.stop();self.broker=None
                self.transport=LocalQueueTransport(self._receive);self.transport.start()
                self.mode=self.transport.name
                print('Fallback aktif: local_queue BUKAN MQTT. Tetap bisa eksperimen Digital Twin.')
        # Pengiriman telemetri, dijalankan di background thread terpisah
        def send_loop():
            # staggered send: N telemetry per interval, bukan satu burst besar.
            spacing=self.cfg.TELEMETRY_INTERVAL_S/len(self.devices)
            deadline=time.monotonic()
            while not self.stop_event.is_set():
                for d in self.devices:
                    if self.stop_event.is_set():break
                    elapsed=time.time()-self.started_at
                    if not (d.offline and elapsed>self.cfg.TELEMETRY_INTERVAL_S*2):
                        payload=d.step(self.cfg.TELEMETRY_INTERVAL_S*self.cfg.TIME_SCALE,
                                       self.cfg.GPS_NOISE_M)
                        try: self.transport.publish(payload);self.sent+=1
                        except Exception as exc: self.publish_errors+=1; self.last_error=str(exc)
                    deadline+=spacing
                    delay=deadline-time.monotonic()
                    if delay>0: self.stop_event.wait(delay)
                    elif delay < -self.cfg.TELEMETRY_INTERVAL_S:deadline=time.monotonic()
        self.thread=threading.Thread(target=send_loop,daemon=True);self.thread.start()
        print('SIMULASI BERJALAN:',self.mode)
    # Menghentikan send loop
    def stop(self):
        self.stop_event.set()
        if self.thread:self.thread.join(timeout=4)
        if self.transport:self.transport.stop()
        if self.broker:self.broker.stop();self.broker=None
        print('SIMULASI DIHENTIKAN. Sent:',self.sent,'Received:',self.engine.snapshot()['count_received'])
    #Mengambil data snapshoot terkini
    def status(self):
        s=self.engine.snapshot(); elapsed=max(time.time()-(self.started_at or time.time()),1e-3)
        return {'mode':self.mode,'sent':self.sent,'received':s['count_received'],
                'avg_sent_per_s':round(self.sent/elapsed,1),'active':s['active'], 'stale':s['stale'],
                'publish_errors':self.publish_errors,'last_error':self.last_error}

SIM=None
# memulai dan menstop simulasi
def start_simulation():
    global SIM
    if SIM is not None:SIM.stop()
    SIM=SimulatorController(CFG)
    SIM.start()
    return SIM

def stop_simulation():
    if SIM is not None:SIM.stop()

print('Controller siap. Untuk mulai, jalankan: start_simulation()')

Controller siap. Untuk mulai, jalankan: start_simulation()


## 9. Dashboard: peta, zone occupancy, dan alerts

Dashboard menampilkan **observed state** dari Digital Twin. Marker biru berarti telemetry masih fresh; merah berarti telemetry stale; kuning berarti pembacaan berada di luar polygon. Batas zona dan kapasitas berbasis konfigurasi simulasi.

Peta dibuat dengan Folium. Klik marker untuk melihat detail perangkat. `show_dashboard()` menampilkan snapshot sekali; widget di bagian berikutnya dapat menjalankan auto-refresh. Map tiles memerlukan Internet; geometri, tabel, dan alert tetap dihitung meskipun tiles tidak termuat.

In [ ]:
import folium
import pandas as pd
# Merepresentasikan data spasial dalam bentuk Peta Poin dan poligon
# Visualisasi Jamaah di CircleMarker
def make_map(snapshot):
    center = xy_to_latlon(500,0)
    # Menggunakan URL tile raster CartoDB Voyager dengan API key milik pengguna
    custom_tile_url = 'https://basemaps.cartocdn.com/rastertiles/voyager/{z}/{x}/{y}.png?key=cb1_55sc_1_b341190e6c4ae0d752001416'
    attribution = '&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors &copy; <a href="https://carto.com/attributions">CARTO</a>'

    m = folium.Map(location=center, zoom_start=15, tiles=custom_tile_url, attr=attribution, control_scale=True)
    for name, z in ZONES.items():
        st = snapshot['zone_stats'][name]
        ratio = st['ratio']
        color = '#c0392b' if ratio > 1 else ('#e6a300' if ratio > .8 else '#2d8a57')
        folium.Polygon(locations=polygon_latlon(z), color=color, weight=2, fill=True,
                       fill_color=color, fill_opacity=.12,
                       tooltip=f'Zone {name}: {st["devices"]}/{st["capacity"]} (training capacity)').add_to(m)
    for s in snapshot['states']:
        color = 'red' if s['stale'] else ('orange' if s['zone']=='OUTSIDE' else 'blue')
        popup = f"{s['device_id']} | Zone {s['zone']} | Seq {s['sequence_no']} | Age {s['state_age_s']:.1f}s | Battery {s['battery_pct']:.1f}%"
        folium.CircleMarker(location=[s['latitude'], s['longitude']], radius=3,
                            color=color, fill=True, fill_opacity=.8, tooltip=popup).add_to(m)
    return m
# Menampilkan dashboad
def show_dashboard(engine=None, show_map=True):
    engine = engine or (SIM.engine if SIM else DT)
    s = engine.snapshot()
    print(f"ACTIVE {s['active']} | STALE {s['stale']} | ACCEPTED {s['count_received']} | INVALID {s['invalid']} | OLD/DUPLICATE {s['old']}")
    if s['latency_p50_ms'] is not None:
        print(f"Telemetry receive latency p50: {s['latency_p50_ms']:.1f} ms | p95: {s['latency_p95_ms']:.1f} ms (same-runtime clocks)")
    print("'Alerts:", "; ".join(s["alerts"]) if s["alerts"] else "None")
    display(pd.DataFrame([{"'Zone'": name, "'Device count'": st["devices"], "'Training capacity'": st["capacity"],
                           "'Occupancy ratio (%)'": round(100*st["ratio"], 1),
                           "'Observed density (device/m²)'": round(st["observed_density_per_m2"], 5)}
                           for name, st in s["zone_stats"].items()]))
    if show_map:
        display(make_map(s))
    return s

print("'Dashboard siap dengan basemap baru (CartoDB Voyager).'")

'Dashboard siap dengan basemap baru (CartoDB Voyager).'


## 10. Interactive controls di Colab

Klik **Start**, lalu **Refresh dashboard**. Untuk pengamatan berulang, aktifkan **Auto-refresh**. Klik **Stop** sebelum menjalankan ulang sel konfigurasi atau melakukan restart runtime. Jika `ipywidgets` belum berhasil dipasang, perintah manual tersedia di bawah.

> **Perhatian:** Auto-refresh map membutuhkan rendering yang lebih berat. Untuk 500 perangkat lebih aman gunakan `Refresh dashboard` manual atau naikkan interval refresh. Tombol `Reset` membuat instance baru, bukan menghapus berkas.

In [ ]:
try:
    import ipywidgets as widgets
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
    WIDGETS_AVAILABLE=True
except ImportError:
    try:
        import ipywidgets as widgets
        WIDGETS_AVAILABLE=True
    except ImportError:WIDGETS_AVAILABLE=False

if WIDGETS_AVAILABLE:
    b_start=widgets.Button(description='Start',button_style='success',icon='play')
    b_stop=widgets.Button(description='Stop',button_style='danger',icon='stop')
    b_refresh=widgets.Button(description='Refresh dashboard',button_style='info',icon='refresh')
    b_reset=widgets.Button(description='Reset',icon='repeat')
    auto=widgets.Checkbox(description='Auto-refresh',value=False)
    dashboard_out=widgets.Output()
    status_out=widgets.Output()
    auto_stop=threading.Event()
    auto_thread=None
    render_lock=threading.Lock()
    def render_once():
        if not render_lock.acquire(blocking=False):return
        try:
            with dashboard_out:
                clear_output(wait=True)
                if SIM: show_dashboard(SIM.engine)
                else: print('Tekan Start terlebih dahulu.')
        finally:render_lock.release()
    def on_start(_):
        with status_out:
            clear_output(wait=True)
            try:start_simulation()
            except Exception as exc:print('Gagal start:',repr(exc))
        render_once()
    def on_stop(_):
        auto.value=False
        auto_stop.set()
        with status_out:
            clear_output(wait=True)
            stop_simulation()
        render_once()
    def on_reset(_):
        on_stop(None)
        with dashboard_out:
            clear_output(wait=True)
            print('Reset selesai; klik Start untuk membuat simulasi baru.')
    def on_auto_change(change):
        global auto_thread
        if not change['new']:
            auto_stop.set();return
        auto_stop.clear()
        def worker():
            while not auto_stop.is_set() and auto.value:
                try:render_once()
                except Exception as exc:
                    with status_out:print('Refresh error:',str(exc))
                auto_stop.wait(CFG.MAP_REFRESH_S)
        auto_thread=threading.Thread(target=worker,daemon=True);auto_thread.start()
    b_start.on_click(on_start); b_stop.on_click(on_stop)
    b_refresh.on_click(lambda _:render_once());b_reset.on_click(on_reset)
    auto.observe(on_auto_change,names='value')
    display(widgets.HBox([b_start,b_stop,b_refresh,b_reset,auto]),status_out,dashboard_out)
else:
    print('ipywidgets tidak tersedia. Gunakan manual: start_simulation(); time.sleep(6); show_dashboard(); stop_simulation()')

Output()

Output()

## 11. Baseline otomatis (tanpa tombol): satu eksperimen singkat

Agar notebook tetap dapat diuji oleh mahasiswa yang mengalami masalah rendering widgets, sel berikut menjalankan simulasi selama beberapa detik dan mencetak metrik. **Jangan menjalankannya ketika simulasi tombol masih aktif** karena dapat terjadi dua sesi dan konflik port broker lokal.

Pertama jalankan `stop_simulation()` jika perlu. Waktu berjalan sesuai jam nyata, bukan simulation time.

In [ ]:
# Jalankan ulang cell ini bila ingin baseline baru. Default 8 detik (bukan 40 detik).
BASELINE_SECONDS=8
stop_simulation()
start_simulation()
time.sleep(BASELINE_SECONDS)
stop_simulation()
print(pd.Series(SIM.status()).to_string())
show_dashboard(SIM.engine,show_map=False)

,'Zone','Device count','Training capacity','Occupancy ratio (%)','Observed density (device/m²)'
0,A,27,45,60.0,0.00068
1,B,27,30,90.0,0.00068
2,C,24,45,53.3,0.00060
3,D,22,45,48.9,0.00055


SIMULASI DIHENTIKAN. Sent: 2224 Received: 2224
SIMULASI DIHENTIKAN. Sent: 2224 Received: 2224
MQTT gagal: RuntimeError: no running event loop
Fallback aktif: local_queue BUKAN MQTT. Tetap bisa eksperimen Digital Twin.
SIMULASI BERJALAN: local_queue (BUKAN MQTT)
SIMULASI DIHENTIKAN. Sent: 401 Received: 401
mode                         local_queue (BUKAN MQTT)
sent                                              401
received                                          401
avg_sent_per_s                                   48.9
active                                            100
stale                                               0
publish_errors                                      0
last_error        RuntimeError: no running event loop
ACTIVE 100 | STALE 0 | ACCEPTED 401 | INVALID 0 | OLD/DUPLICATE 0
Telemetry receive latency p50: 0.2 ms | p95: 0.7 ms (same-runtime clocks)
'Alerts: None


,'Zone','Device count','Training capacity','Occupancy ratio (%)','Observed density (device/m²)'
0,A,21,45,46.7,0.00052
1,B,28,30,93.3,0.00070
2,C,31,45,68.9,0.00077
3,D,20,45,44.4,0.00050


{'states': [{'device_id': 'WB-001',
   'sequence_no': 5,
   'latitude': 21.41352886,
   'longitude': 39.90275938,
   'x': 907.7801459822787,
   'y': -52.44730480021417,
   'zone': 'D',
   'battery_pct': 86.2,
   'event_ts': 1794289925.665282,
   'received_ts': 1794289925.665431,
   'accuracy_m': 3.0,
   'stale': False,
   'state_age_s': 0.2041318416595459},
  {'device_id': 'WB-002',
   'sequence_no': 4,
   'latitude': 21.41443704,
   'longitude': 39.89897033,
   'x': 515.1011707425436,
   'y': 48.65129279975278,
   'zone': 'C',
   'battery_pct': 85.33,
   'event_ts': 1794289923.685326,
   'received_ts': 1794289923.6855109,
   'accuracy_m': 3.0,
   'stale': False,
   'state_age_s': 2.184051990509033},
  {'device_id': 'WB-003',
   'sequence_no': 4,
   'latitude': 21.41436991,
   'longitude': 39.89656666,
   'x': 265.9963364400354,
   'y': 41.17838119998467,
   'zone': 'B',
   'battery_pct': 91.47,
   'event_ts': 1794289923.705324,
   'received_ts': 1794289923.7055106,
   'accuracy_m': 3.

## 12. Eksperimen 1: Device count dan telemetry interval

**Pertanyaan:** apa perbedaan theoretical message rate dan actual message rate?

1. Catat rata-rata pesan/detik dengan `N_DEVICES=100`, `INTERVAL=2` detik.
2. Ulangi dengan `N_DEVICES=20` dan `500`, dengan interval sama.
3. Ulangi dengan interval 1 dan 5 detik.
4. Bandingkan `sent` dan `received`. Bila broker memakai QoS 0, selisih tidak otomatis berarti bug di Digital Twin.

**Prediksi yang diuji:** \(R_{expected} = N / \Delta t\) pesan per detik, jika tiap perangkat mengirim satu pesan setiap interval. Gunakan **steady-state** dan abaikan awal/akhir proses untuk estimasi throughput yang lebih teliti.

In [ ]:
# Hitung perencanaan message rate sebelum menjalankan percobaan.
rate_design = pd.DataFrame([{'devices':n,'interval_s':dt,'expected_msg_s':n/dt,
                             'expected_msg_min':60*n/dt} for n in (20,100,500) for dt in (1,2,5)])
display(rate_design)
# Ubah CFG.N_DEVICES dan CFG.TELEMETRY_INTERVAL_S pada sel Configuration,
# kemudian jalankan ulang Simulator Controller dengan start_simulation().

,devices,interval_s,expected_msg_s,expected_msg_min
0,20,1,20.0,1200.0
1,20,2,10.0,600.0
2,20,5,4.0,240.0
3,100,1,100.0,6000.0
4,100,2,50.0,3000.0
5,100,5,20.0,1200.0
6,500,1,500.0,30000.0
7,500,2,250.0,15000.0
8,500,5,100.0,6000.0


## 13. Eksperimen 2: State synchronization dan penanganan pesan terlambat

**Pertanyaan:** mengapa Digital Twin harus menyimpan `sequence_no`, `received_ts`, dan `event_ts`?

Berikut uji deterministik tanpa broker. Kita kirim pesan terbaru, lalu sebuah pesan lama dan satu duplikat. Hanya pesan pertama yang boleh mengubah state.

In [ ]:
test_dt=DigitalTwinEngine(CFG)
t0=datetime.now(timezone.utc).isoformat()
base={'device_id':'WB-TEST','timestamp':t0,'latitude':ORIGIN_LAT,'longitude':ORIGIN_LON,
      'battery_pct':90,'status':'ACTIVE'}
assert test_dt.ingest(dict(base,sequence_no=2))
assert not test_dt.ingest(dict(base,sequence_no=1))
assert not test_dt.ingest(dict(base,sequence_no=2))
assert test_dt.snapshot()['states'][0]['sequence_no']==2
assert test_dt.snapshot()['old']==2
print('PASS: pesan out-of-order dan duplicate ditolak.')

PASS: pesan out-of-order dan duplicate ditolak.


## 14. Eksperimen 3: Spatial context, GPS noise, dan geofencing

**Pertanyaan:** apakah sebuah koordinat tepat di tepi geofence cukup kuat untuk menyimpulkan seseorang berpindah zona?

Ubah noise pengukuran dari 0 sampai 15 meter. Perangkat asli pada `x=245` meter berada 5 meter sebelum batas A/B. Noise membuat perangkat terdeteksi di dua zona yang berbeda meskipun posisi sebenarnya sama. Ini menunjukkan mengapa geofence nyata biasanya memerlukan *hysteresis*, confidence region, atau beberapa pembacaan.

In [ ]:
def boundary_noise_experiment(noise_m, trials=1000, seed=7):
    r=random.Random(seed)
    detected=[point_zone(245+r.gauss(0,noise_m),0) for _ in range(trials)]
    return {'gps_noise_m':noise_m,'zone_A':detected.count('A'),
            'zone_B':detected.count('B'),'outside':detected.count('OUTSIDE')}
display(pd.DataFrame([boundary_noise_experiment(n) for n in (0,1,3,5,10,15)]))

,gps_noise_m,zone_A,zone_B,outside
0,0,1000,0,0
1,1,1000,0,0
2,3,956,44,0
3,5,850,150,0
4,10,676,324,0
5,15,613,387,0


## 15. Eksperimen 4: Occupancy, crowd density, dan threshold

**Pembedaan konsep:** jumlah wristband di zona adalah **observed device count**. Itu bukan otomatis jumlah orang sebenarnya. Jika satu orang membawa dua perangkat, atau sebagian orang tidak mempunyai perangkat, estimasi dapat bias. Perhitungan `device/m²` tidak otomatis dapat diterjemahkan menjadi risiko keselamatan atau batas kepadatan yang valid secara ilmiah.

**Pertanyaan mahasiswa:** apa yang terjadi jika kapasitas zona B diubah dari 30 menjadi 20 atau 60? Bedakan perubahan *angka alert* dari perubahan situasi fisik.

In [ ]:
def check_capacity_example(count, capacities=(20,30,60)):
    return pd.DataFrame([{'devices_in_B':count, 'training_capacity':cap,
                          'occupancy_ratio':round(count/cap,3),
                          'alert':count/cap>1} for cap in capacities])
display(check_capacity_example(count=42))

,devices_in_B,training_capacity,occupancy_ratio,alert
0,42,20,2.1,True
1,42,30,1.4,True
2,42,60,0.7,False


## 16. Eksperimen 5: Scenario injection dan Decision Support

1. Set `CFG.SCENARIO='crowded_B'` dan ulangi baseline. Bandingkan occupancy pada setiap zona.
2. Set `CFG.SCENARIO='device_offline'`, jalankan lebih dari `STALE_AFTER_S`, dan amati perbedaan **stale devices** versus jumlah device yang benar-benar bergerak pada simulator.
3. Set `GPS_NOISE_M=12` lalu amati *false geofence alerts*.
4. Diskusikan rekomendasi: apa data tambahan (kamera, petugas, arus masuk/keluar) yang diperlukan sebelum mengubah jalur kerumunan?

**Jangan menganggap alarm adalah kepastian bahaya.** Threshold pada notebook adalah parameter latihan, bukan batas keselamatan yang tervalidasi untuk Haji.

In [ ]:
# Eksperimen deterministik: menguji pembuatan alert.
alert_dt=DigitalTwinEngine(CFG)
now=datetime.now(timezone.utc).isoformat()
for i in range(36):
    lat,lon=xy_to_latlon(350+(i%6)*2,(i//6)*2)
    payload={'device_id':f'WB-{i+1:03d}','timestamp':now,'sequence_no':1,
             'latitude':lat,'longitude':lon,'battery_pct':95,'status':'ACTIVE'}
    alert_dt.ingest(payload)
s=alert_dt.snapshot()
assert s['zone_stats']['B']['devices']==36
assert any('Zone B' in a for a in s['alerts'])
print('PASS: crowd alert muncul untuk Zone B:',s['alerts'])

PASS: crowd alert muncul untuk Zone B: ['CHECK OCCUPANCY: Zone B > training capacity (36/30)']


## 17. Performance evaluation dan ekspor CSV

Data yang dicatat: acceptance count, duplicate count, state freshness, penerimaan pesan, latency dari waktu pembuatan payload, dan perpindahan zona. Metrik diuji pada simulasi, bukan benchmark skala besar. Data diekspor sebagai CSV untuk evaluasi di Excel.

Perlu diperhatikan: timestamp dibuat di runtime yang sama dengan penerimaan subscriber. Karena itu, angka latency tidak mencakup kesalahan sinkronisasi clock antar perangkat fisik, radio propagation, dan jaringan produksi.

In [ ]:
from pathlib import Path
EXPORT_DIR=Path('/content/hajj_dt_outputs') if Path('/content').exists() else Path('/mnt/data/hajj_dt_outputs')
EXPORT_DIR.mkdir(exist_ok=True,parents=True)
export_engine=SIM.engine if SIM is not None else alert_dt
snapshot=export_engine.snapshot()
with export_engine.lock:
    events=pd.DataFrame(list(export_engine.events))
    transitions=pd.DataFrame(list(export_engine.transitions))
states=pd.DataFrame(snapshot['states'])
zones=pd.DataFrame([{'zone':z,**v} for z,v in snapshot['zone_stats'].items()])
frames={'telemetry_events':events,'device_states':states,'zone_summary':zones,'zone_transitions':transitions}
for name,df in frames.items():
    path=EXPORT_DIR/f'{name}.csv'
    df.to_csv(path,index=False)
    print(name,':',len(df),'rows =>',path)
print('Metrik akhir:',{k:snapshot[k] for k in ('active','stale','count_received','invalid','old','latency_p50_ms','latency_p95_ms')})

telemetry_events : 401 rows => /content/hajj_dt_outputs/telemetry_events.csv
device_states : 100 rows => /content/hajj_dt_outputs/device_states.csv
zone_summary : 4 rows => /content/hajj_dt_outputs/zone_summary.csv
zone_transitions : 61 rows => /content/hajj_dt_outputs/zone_transitions.csv
Metrik akhir: {'active': 100, 'stale': 0, 'count_received': 401, 'invalid': 0, 'old': 0, 'latency_p50_ms': 0.19240379333496094, 'latency_p95_ms': 0.5424022674560547}


## 18. Automatic tests: validasi sebelum mahasiswa melakukan demo

Uji berikut tidak memerlukan jaringan atau broker. Tujuan pengujian adalah memastikan fungsi geospatial, state synchronization, geofence, serta alert berjalan dengan benar. **Pengujian MQTT end-to-end tetap harus dibuktikan dengan melihat `SIM.mode` mengandung `MQTT` dan `received > 0`.**

In [ ]:
def run_unit_tests():
    # 1. Coordinate conversion round-trip
    for x,y in [(0,0),(100,20),(750,-60),(999,79)]:
        lat,lon=xy_to_latlon(x,y)
        xx,yy=latlon_to_xy(lat,lon)
        assert abs(xx-x)<1e-5 and abs(yy-y)<1e-5
    # 2. Point-in-zone and boundaries
    assert point_zone(125,0)=='A'
    assert point_zone(375,0)=='B'
    assert point_zone(625,0)=='C'
    assert point_zone(875,0)=='D'
    assert point_zone(125,90)=='OUTSIDE'
    # 3. Schema + sequencing
    e=DigitalTwinEngine(CFG)
    p={'device_id':'WB-777','timestamp':datetime.now(timezone.utc).isoformat(),
       'sequence_no':1,'latitude':ORIGIN_LAT,'longitude':ORIGIN_LON,'battery_pct':50,'status':'ACTIVE'}
    assert e.ingest(p)
    assert not e.ingest(p)
    assert not e.ingest(dict(p,device_id='broken'))
    assert e.snapshot()['count_received']==1
    assert e.snapshot()['old']==1
    assert e.snapshot()['invalid']==1
    # 4. Stale detection without waiting
    old_time=time.time()+CFG.STALE_AFTER_S+1
    assert e.snapshot(now=old_time)['stale']==1
    # 5. Expected telemetry rate calculation
    assert 100/2 == 50
    # 6. Alert production
    assert any('Zone B' in a for a in alert_dt.snapshot()['alerts'])
    return '6/6 automatic tests PASS'
print(run_unit_tests())
if SIM is not None:
    print('Transport:',SIM.mode,'| Received:',SIM.engine.snapshot()['count_received'])
    if 'MQTT' not in SIM.mode:
        print('PERHATIAN: end-to-end MQTT BELUM diverifikasi pada sesi ini; mode yang berjalan adalah', SIM.mode)

6/6 automatic tests PASS
Transport: local_queue (BUKAN MQTT) | Received: 401


## 19. Tugas mahasiswa: pertanyaan yang harus dijawab dari data

**Jawab menggunakan hasil eksekusi, bukan definisi saja.** Lampirkan screenshot dashboard dan CSV yang dihasilkan.

| No. | Pertanyaan | Bukti yang diminta |
|---|---|---|
| 1 | Bandingkan pengiriman pada 20, 100, dan 500 wristband. Apakah message rate mendekati teori? | Tabel expected vs measured |
| 2 | Apa yang terjadi jika telemetry interval 2 s diubah menjadi 5 s? | State age dan message count |
| 3 | Mengapa sequence number penting untuk synchronization? | Uji duplicate/out-of-order |
| 4 | Bagaimana noise GPS memengaruhi geofencing? | Tabel hasil noise experiment |
| 5 | Apa perbedaan device count, occupancy ratio, dan people/m²? | Analisis asumsi dan keterbatasan |
| 6 | Bagaimana membedakan perangkat offline dari jemaah yang tidak bergerak? | Screenshot status STALE |
| 7 | Mengapa sistem ini disebut Digital Twin prototype dan bukan sekadar tracking map? | Penjelasan model entitas, hubungan, dan synchronization |
| 8 | Apa batasan Colab dan broker publik untuk deployment nyata? | Evaluasi privacy, scale, reliability |

**Pengembangan opsional:** prediksi occupancy 30 detik ke depan menggunakan moving average, tetapi jangan mengklaim prediksi tervalidasi jika belum melakukan evaluasi forecasting. Model 3D boleh ditambahkan jika memang ada konteks vertikal yang memerlukan 3D.

## 20. Project berikutnya: Smart Event Wristband

Pada sesi berikutnya, mahasiswa mendemonstrasikan prototipe untuk **konser, stadion, festival, atau event kampus**. Data harus berupa simulasi atau diperoleh dengan izin, bukan data pribadi pengunjung yang disebarkan ke broker publik.

**Minimum live demo:**
1. Minimal 20 virtual wristbands atau perangkat nyata yang terhubung.
2. Transport message yang dapat dijelaskan dan dibuktikan (MQTT untuk proyek MQTT).
3. Real-time Digital Twin state, minimal dua zona dan geofencing.
4. Dashboard peta dan minimal satu alert.
5. Pengujian duplicate/out-of-order, disconnect, serta satu ukuran performance.
6. Penjelasan perbedaan IoT telemetry, Digital Twin, dan DSS.

**Kesimpulan:** IoT mengamati, transport mengirim, Digital Twin memberikan model keadaan dan konteks spatial, analytics mengolah data, serta DSS membantu operator mengidentifikasi kondisi yang perlu diperiksa.

### Referensi
- Materi **PB10MAT_TIC06 Topic 9**, *IoT in Large Scale Events*, Universitas Bunda Mulia.
- Greg Biegel, *Digital Twins in Action*, Manning (2026), khususnya Bab 1–7, 8–10.
- Eclipse Paho MQTT Python client: https://eclipse.dev/paho/files/paho.mqtt.python/html/
- AMQTT broker: https://amqtt.readthedocs.io/
- Folium: https://python-visualization.github.io/folium/
- Google Colab FAQ: https://research.google.com/colaboratory/faq.html

**Penutupan runtime:** `stop_simulation()` untuk menghentikan publisher/subscriber sebelum menutup Colab.